## Step 0: Load libraries

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Step 1: Load and Clean pre-2026 data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result: 157,821 → 114,147, dropped 43,674)
2. **Drop duplicates** — remove rows with duplicated data
3. Create the log-transformed target variable `log_resale_price_real`

In [17]:
df_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_pre2026.csv")
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df)
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

Initial shape: (134479, 37)
After dropping nulls: (134301, 37) (dropped 178)
After dropping duplicates: (134211, 37) (dropped 90)


## Step 2: Stratified Train/Validation Split (80/20)

In [18]:
import re

# Target variable: log_resale_price_real
# RPI-adjusted prices are used so the model learns purely from flat characteristics
# rather than market-wide price appreciation over time, since year is not a model feature.
target = "log_resale_price_real"


from sklearn.model_selection import train_test_split

df["strat_key"] = (df["town"].astype(str) + "_" +
                   df["flat_type"].astype(str) + "_" +
                   df["year"].astype(str))

strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index
n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)]
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["strat_key"])
print(f"Train size: {len(train_df):,} | Test size: {len(val_df):,}")

print("\nYear distribution (%):")
train_year = train_df["year"].value_counts(normalize=True).sort_index().rename("Train %")
test_year = val_df["year"].value_counts(normalize=True).sort_index().rename("Validation %")
year_dist = pd.concat([train_year, test_year], axis=1)
print(year_dist.applymap(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
train_flat = train_df["flat_type"].value_counts(normalize=True).rename("Train %")
test_flat = val_df["flat_type"].value_counts(normalize=True).rename("Validation %")
flat_dist = pd.concat([train_flat, test_flat], axis=1)
print(flat_dist.applymap(lambda x: f"{x:.2%}"))

Dropped 6 rows with singleton strat_key combinations. Remaining: 134,205
Train size: 107,364 | Test size: 26,841

Year distribution (%):
     Train % Validation %
year                     
2021  21.61%       21.61%
2022  19.86%       19.87%
2023  19.15%       19.16%
2024  20.71%       20.70%
2025  18.67%       18.67%

Flat type distribution (%):
                 Train % Validation %
flat_type                            
4 ROOM            43.02%       43.00%
5 ROOM            24.25%       24.23%
3 ROOM            23.71%       23.71%
EXECUTIVE          6.62%        6.63%
2 ROOM             2.35%        2.36%
1 ROOM             0.03%        0.03%
MULTI-GENERATION   0.03%        0.03%


## Step 3: Train Random Forest Model
**Keep feature set aligned with baseline OLS model**

In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# =========================================
# Features
# =========================================
feature_cols = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km",
    "dist_nearest_park_m", "num_parks_1km", "dist_nearest_sportsg_m",
    "dist_nearest_mall_m", "dist_nearest_healthcare_m",
    "flat_type", "town", "floor_category"
]

categorical_features = ["flat_type", "town", "floor_category"]

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()

X_train[categorical_features] = enc.fit_transform(train_df[categorical_features])
X_val[categorical_features]   = enc.transform(val_df[categorical_features])

# =========================================
# Targets — clearly named, consistently used
# =========================================
y_train_log = train_df[target].values          # log scale, for model training
y_val_log   = val_df[target].values

y_val_actual = val_df["resale_price_real"].values   # dollar scale, for RMSE

# =========================================
# Optuna Objective
# =========================================
def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 800),
        "max_depth":         trial.suggest_categorical("max_depth", [None, 10, 20, 30, 50]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 4),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train_log)             # train on log scale

    preds_log = model.predict(X_val)
    rmse      = np.sqrt(mean_squared_error(y_val_log, preds_log))  

    return rmse

# =========================================
# Run Tuning
# =========================================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)         

print("Best params:", study.best_params)
print(f"Best RMSE:  ${study.best_value:,.0f}")

# =========================================
# Final Model
# =========================================
rf_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train_log)

[I 2026-04-04 23:28:49,021] A new study created in memory with name: no-name-f63040da-e8d7-4e57-8dc4-cfc76e628455
[I 2026-04-04 23:29:06,559] Trial 0 finished with value: 0.08800453543558681 and parameters: {'n_estimators': 174, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 0.8}. Best is trial 0 with value: 0.08800453543558681.
[I 2026-04-04 23:29:12,072] Trial 1 finished with value: 0.05661629226746903 and parameters: {'n_estimators': 107, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 0.05661629226746903.
[I 2026-04-04 23:29:44,957] Trial 2 finished with value: 0.05542279171958182 and parameters: {'n_estimators': 211, 'max_depth': 30, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 0.8}. Best is trial 2 with value: 0.05542279171958182.
[I 2026-04-04 23:30:32,898] Trial 3 finished with value: 0.055818777599904924 and parameters: {'n_estimators': 356, 'max_depth': 30,

Best params: {'n_estimators': 394, 'max_depth': 50, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 0.5}
Best RMSE:  $0


,n_estimators,394
,criterion,'squared_error'
,max_depth,50
,min_samples_split,7
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Step 4: Generating and Evaluating Predictions for Random Forest Model

In [24]:
# =========================================
# Predictions
# =========================================
y_val_pred_log = rf_model.predict(X_val)
y_val_pred = np.exp(y_val_pred_log)   # convert back to price

# =========================================
# Evaluation 
# =========================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    errors = np.array(y_true) - np.array(y_pred)
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

validation_rmse = rmse(y_val_actual, y_val_pred)
validation_linlin = linlin_loss(y_val_actual, y_val_pred, underpredict_weight=2.0)
r2 = r2_score(y_val_actual, y_val_pred)

print("=== RANDOM FOREST MODEL EVALUATION ===")
print(f"R^2: {r2:.3f}")
print(f"RMSE: ${validation_rmse:,.0f}")
print(f"Linlin Loss: ${validation_linlin:,.0f}")


=== RANDOM FOREST MODEL EVALUATION ===
R^2: 0.964
RMSE: $39,641
Linlin Loss: $41,313


## Step 5: Load Test Data (2026 data)

In [25]:
df_test_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_2026.csv")
print(f"Initial shape: {df_test_raw.shape}")

df_test = df_test_raw.dropna()
print(f"After dropping nulls: {df_test.shape} (dropped {len(df_test_raw) - len(df_test)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df_test)
df_test = df_test.drop_duplicates()
print(f"After dropping duplicates: {df_test.shape} (dropped {n_before - len(df_test)})")

df_test['log_resale_price_real'] = np.log(df_test['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

Initial shape: (6058, 37)
After dropping nulls: (6055, 37) (dropped 3)
After dropping duplicates: (6051, 37) (dropped 4)


## Step 6: Check performance of Random Forest model on 2026 test data

In [26]:
# =========================================
# Build feature matrix
# =========================================
X_test = df_test[feature_cols].copy()
X_test[categorical_features] = enc.transform(df_test[categorical_features])

# =========================================
# Predictions
# =========================================
y_test_pred_log = rf_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log)

# =========================================
# Actual values
# =========================================
y_test_actual = df_test['resale_price_real']

# =========================================
# Evaluation
# =========================================
rmse_test = rmse(y_test_actual, y_test_pred)
linlin_test = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
r2_test = r2_score(y_test_actual, y_test_pred)

print("=== 2026 TEST SET (OUT-OF-SAMPLE) ===")
print(f"R^2: {r2_test:.3f}")
print(f"RMSE: ${rmse_test:,.0f}")
print(f"Linlin Loss: ${linlin_test:,.0f}")



=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.961
RMSE: $41,723
Linlin Loss: $42,641


## Step 7: Train extended Random Forest Model with additional feature
**Additional feature added - "dist_cbd_m"**

In [27]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# =========================================
# Features 
# =========================================
feature_cols_extra = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km",
    "dist_nearest_park_m", "num_parks_1km", "dist_nearest_sportsg_m",
    "dist_nearest_mall_m", "dist_nearest_healthcare_m", "dist_cbd_m", 
    "flat_type", "town", "floor_category"
]

continuous_features_extra  = feature_cols_extra[:11]
categorical_features_extra = ["flat_type", "town", "floor_category"]

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train_extra = train_df[feature_cols_extra].copy()
X_val_extra   = val_df[feature_cols_extra].copy()

X_train_extra[categorical_features_extra] = enc.fit_transform(train_df[categorical_features_extra])
X_val_extra[categorical_features_extra]   = enc.transform(val_df[categorical_features_extra])

y_train_extra  = train_df[target].values        # log scale, for training
y_val_extra    = val_df[target].values

y_val_actual   = val_df["resale_price_real"].values   # dollar scale, for RMSE

# =========================================
# Optuna Objective
# =========================================
def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 800),
        "max_depth":         trial.suggest_categorical("max_depth", [None, 10, 20, 30, 50]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 4),
        "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train_extra, y_train_extra)      # train on log scale

    preds_log_extra = model.predict(X_val_extra)
    rmse_extra      = np.sqrt(mean_squared_error(y_val_extra, preds_log_extra))  

    return rmse_extra

# =========================================
# Run Tuning
# =========================================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)
print(f"Best RMSE:  ${study.best_value:,.0f}")

# =========================================
# Final Model
# =========================================
rf_model_extra = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

rf_model_extra.fit(X_train_extra, y_train_extra)

[I 2026-04-04 23:58:14,491] A new study created in memory with name: no-name-d3dc4158-bb7c-4472-94c3-3c68259c061e
[I 2026-04-04 23:58:27,047] Trial 0 finished with value: 0.092429923633577 and parameters: {'n_estimators': 460, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.092429923633577.
[I 2026-04-05 00:00:16,004] Trial 1 finished with value: 0.05446377936047129 and parameters: {'n_estimators': 699, 'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 0.8}. Best is trial 1 with value: 0.05446377936047129.
[I 2026-04-05 00:00:45,038] Trial 2 finished with value: 0.05586646931092186 and parameters: {'n_estimators': 546, 'max_depth': None, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.05446377936047129.
[I 2026-04-05 00:00:56,756] Trial 3 finished with value: 0.055389448163035426 and parameters: {'n_estimators': 224, 'max_depth': Non

Best params: {'n_estimators': 614, 'max_depth': 50, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': 0.5}
Best RMSE:  $0


,n_estimators,614
,criterion,'squared_error'
,max_depth,50
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Step 8: Generate and Evaluate Predictions for extended Random Forest model

In [28]:
# =========================================
# Predictions
# =========================================
y_val_pred_log_extra = rf_model_extra.predict(X_val_extra)
y_val_pred_extra = np.exp(y_val_pred_log_extra)   # convert back to price

# =========================================
# Evaluation 
# =========================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    errors = np.array(y_true) - np.array(y_pred)
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

validation_rmse = rmse(y_val_actual, y_val_pred_extra)
validation_linlin = linlin_loss(y_val_actual, y_val_pred_extra, underpredict_weight=2.0)
r2 = r2_score(y_val_actual, y_val_pred_extra)

print("=== RANDOM FOREST MODEL EVALUATION ===")
print(f"R^2: {r2:.3f}")
print(f"RMSE: ${validation_rmse:,.0f}")
print(f"Linlin Loss: ${validation_linlin:,.0f}")

=== RANDOM FOREST MODEL EVALUATION ===
R^2: 0.965
RMSE: $38,860
Linlin Loss: $40,688


## Step 9: Check performance of extended Random Forest model on 2026 test data

In [30]:
# =========================================
# Build feature matrix
# =========================================
X_test_extra = df_test[feature_cols_extra].copy()
X_test_extra[categorical_features_extra] = enc.transform(df_test[categorical_features_extra])

# =========================================
# Predictions
# =========================================
y_test_pred_log_extra = rf_model_extra.predict(X_test_extra)
y_test_pred_extra = np.exp(y_test_pred_log_extra)

# =========================================
# Actual values
# =========================================
y_test_actual_extra = df_test['resale_price_real']

# =========================================
# Evaluation
# =========================================
rmse_test = rmse(y_test_actual_extra, y_test_pred_extra)
linlin_test = linlin_loss(y_test_actual_extra, y_test_pred_extra, underpredict_weight=2.0)
r2_test = r2_score(y_test_actual_extra, y_test_pred_extra)

print("=== 2026 TEST SET (OUT-OF-SAMPLE) ===")
print(f"R^2: {r2_test:.3f}")
print(f"RMSE: ${rmse_test:,.0f}")
print(f"Linlin Loss: ${linlin_test:,.0f}")



=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.965
RMSE: $39,484
Linlin Loss: $40,529


## Step 10: Conformal Prediction Intervals

Conformal prediction gives us a rigorous way to wrap uncertainty around our point estimates. The key guarantee: if the 2026 data comes from the same distribution as our validation set, then (1 - alpha)% of our intervals will contain the true price.**

We use the validation split as our "calibration set" — data the model has never seen during training, so its errors are honest.**

In [31]:
# ===================================================================
# Step 1: Calibration residuals (in actual price space, after exp) 
# ===================================================================
# SIGNED residuals (true - pred), not absolute. Sign matters because we treat underprediction and overprediction differently.
# We compute these AFTER exp(), in actual dollar space.
# If we worked in log space, our intervals would be multiplicative and much harder to interpret or apply to new predictions.

cal_residuals = y_val_actual - y_val_pred_extra   # true - pred

# ===================================================================
# Step 2: Asymmetric quantiles to match linlin loss 
# ===================================================================
# Because underprediction costs 2x more (underpredict_weight=2.0), we split the alpha "miss budget" unequally between the two tails.
# The intuition: if we have 5% of predictions allowed to miss, and underprediction is twice as bad, we protect the upper side twice as hard — giving it only 1/(1+2) = 33% of the miss budget.
# upper tail gets: alpha * 1/(1+w)    = 0.05 * 0.333 = 1.67% tolerance
# lower tail gets: alpha * w/(1+w)    = 0.05 * 0.667 = 3.33% tolerance
# This means our upper bound is held very tight (only 1.67% of true prices are allowed to exceed it), while the lower bound is looser.
alpha = 0.05   # 95% overall coverage
w = 2.0        # underpredict_weight

alpha_upper = alpha * (1 / (1 + w))    # 0.0167 — upper tail tolerance
alpha_lower = alpha * (w / (1 + w))    # 0.0333 — lower tail tolerance

n = len(cal_residuals)

# Finite-sample correction: use (n/(n+1)) instead of plain quantile.
# Without this, coverage is only guaranteed asymptotically (large n).
# With it, we get exact finite-sample coverage — always include this.
q_low  = np.quantile(cal_residuals, alpha_lower * (n / (n + 1)))   # negative: how much we can overpredict
q_high = np.quantile(cal_residuals, 1 - alpha_upper * (n / (n + 1)))  # positive: how much we can underpredict

print(f"\n=== CONFORMAL PREDICTION SETUP ===")
print(f"Calibration samples (n): {n}")
print(f"Lower quantile (q_low):  ${q_low:,.0f}  (model can overpredict by this much)")
print(f"Upper quantile (q_high): ${q_high:,.0f}  (model can underpredict by this much)")
print(f"Typical interval width:  ${q_high - q_low:,.0f}")

# --- Step 3: Diagnosis on validation set ---
# Before applying to 2026, sanity-check that the intervals behave correctly on the calibration set itself.
# Expected: empirical coverage ≈ 95%, upper misses ≈ 1.67%, lower misses ≈ 3.33%.
lower_val = y_val_pred_extra + q_low    # q_low is negative, so this subtracts
upper_val = y_val_pred_extra + q_high

covered_val = (
    (y_val_actual >= lower_val) &
    (y_val_actual <= upper_val)
)

# Count directional misses separately — they mean different things.
# Upper misses = we underpredicted (costly, per our linlin loss).
# Lower misses = we overpredicted (less costly, so we allow more of these).
underpredict_misses = (y_val_actual > upper_val).sum()
overpredict_misses  = (y_val_actual < lower_val).sum()

print(f"\n=== CALIBRATION DIAGNOSTICS ===")
print(f"Empirical coverage:       {covered_val.mean():.2%}  (target: {1-alpha:.0%})")
print(f"Underpredict misses:      {underpredict_misses} ({underpredict_misses/n:.2%})  (target: {alpha_upper:.2%})")
print(f"Overpredict misses:       {overpredict_misses} ({overpredict_misses/n:.2%})  (target: {alpha_lower:.2%})")


# =========================================
# Apply conformal intervals to 2026 OOS Data
# =========================================
# q_low and q_high are fixed constants from the calibration step above.
# Every 2026 prediction gets the same additive shift applied.
# The interval is intentionally asymmetric: wider above the point estimate (by q_high) than below (by |q_low|), because we penalise underprediction more.
lower_oos = y_test_pred_extra + q_low
upper_oos = y_test_pred_extra + q_high

covered_oos = (
    (y_test_actual >= lower_oos) &
    (y_test_actual <= upper_oos)
)

under_oos = (y_test_actual > upper_oos).sum()
over_oos  = (y_test_actual < lower_oos).sum()
n_oos     = len(y_test_actual)


print(f"\n=== OOS (2026) CONFORMAL INTERVAL PERFORMANCE ===")
# If OOS coverage is well below 95%: distribution shift between <=2025 and 2026 (e.g. cooling measures, rate changes). 
# If OOS coverage is well above 95%: intervals are too conservative (validation residuals were larger than typical OOS errors). Could tighten alpha to 0.08-0.10 and still have valid coverage.
print(f"Empirical coverage:   {covered_oos.mean():.2%}  (target: 95%)")
print(f"Underpredict misses:  {under_oos} ({under_oos/n_oos:.2%})  (target: {alpha_upper:.2%})")
print(f"Overpredict misses:   {over_oos} ({over_oos/n_oos:.2%})  (target: {alpha_lower:.2%})")


# --- Final output dataframe ---
results_oos = pd.DataFrame({
    'actual_price':  y_test_actual.values,
    'pred_price':    y_test_pred_extra,
    'lower_95':      lower_oos,
    'upper_95':      upper_oos,
    'covered':       covered_oos,
    'residual':      y_test_actual.values - y_test_pred_extra
})

print(f"\n=== SAMPLE PREDICTIONS WITH INTERVALS ===")
print(results_oos.head(10).to_string(index=False))


=== CONFORMAL PREDICTION SETUP ===
Calibration samples (n): 26841
Lower quantile (q_low):  $-62,823  (model can overpredict by this much)
Upper quantile (q_high): $97,136  (model can underpredict by this much)
Typical interval width:  $159,960

=== CALIBRATION DIAGNOSTICS ===
Empirical coverage:       95.00%  (target: 95%)
Underpredict misses:      448 (1.67%)  (target: 1.67%)
Overpredict misses:       895 (3.33%)  (target: 3.33%)

=== OOS (2026) CONFORMAL INTERVAL PERFORMANCE ===
Empirical coverage:   93.89%  (target: 95%)
Underpredict misses:  110 (1.82%)  (target: 1.67%)
Overpredict misses:   260 (4.30%)  (target: 3.33%)

=== SAMPLE PREDICTIONS WITH INTERVALS ===
 actual_price    pred_price      lower_95      upper_95  covered      residual
     345000.0 321659.388956 258836.019817 418795.877478     True  23340.611044
     325000.0 321657.075778 258833.706640 418793.564300     True   3342.924222
     330000.0 324074.498884 261251.129745 421210.987406     True   5925.501116
     658

## Step 11: Save model artifacts
Serialize the fitted RF model, conformal quantiles, and feature column list so they can be loaded without re-fitting.

In [36]:
import joblib, json

class LogToPriceWrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        log_pred = self.model.predict(X)
        return np.exp(log_pred)   # convert back

# wrap model
rf_model_actual = LogToPriceWrapper(rf_model_extra)

# Save the model 
joblib.dump(rf_model_actual, "rf_model.pkl")

# Save the OrdinalEncoder (needed for inference on new data)
joblib.dump(enc, "rf_encoder.pkl")

# Save the feature column order
joblib.dump(list(X_train_extra.columns), "rf_feature_cols.pkl", compress=3)

# Save the two conformal quantiles (q_low, q_high) computed from the calibration step
with open("rf_conformal_quantiles.json", "w") as f:
    json.dump({"q_low": float(q_low), "q_high": float(q_high)}, f)